### Introduction
This is the main data analysis notebook, here I will clean the data and apply natural language processing techniques to extract the years of experience and the skills from the job descriptions. After which I will do a simple frequency count to see which among the two BI tools, Power BI or Tableau is most mentioned in job descricptions.

### Data wrangling

In [35]:
# Import all necessary libraries

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns


In [36]:
#clean the first dataset from glassdoor

df1=pd.read_csv(r'C:\Users\Wambui\Desktop\Job_Market_Analysis-BI_Tools\glassdoor_data.csv')
df1.head()

,title,description,link
0,Tableau Developer,Job Title: Tableau Developer\nDescription: The...,https://www.glassdoor.com/job-listing/tableau-...
1,Power BI Co-Op,"About American Ordnance\nAt American Ordnance,...",https://www.glassdoor.com/job-listing/power-bi...
2,Power BI Designer,We are looking for a Microsoft PowerBI Designe...,https://www.glassdoor.com/job-listing/power-bi...
3,Power BI Developer,"19562\nIES Holdings\nSugar Land, Texas\n\nJob ...",https://www.glassdoor.com/job-listing/power-bi...
4,Lead Power BI Developer,We are seeking a Power BI Reporting Lead to dr...,https://www.glassdoor.com/job-listing/lead-pow...


In [37]:
#handle missingness, drop any row which missing values
df1.dropna(inplace=True)

For this analysis, I also want to examine if the preferrence for Power BI or Tableau is based on the specific data role, for example Data Analyst or Data Manager. We know the same job role, technically can be referred to in a myriad of ways, depending on the company. So the next step is to categorize the scrapped job titles into what I am calling 'core titles'. This way a 'Sales BI Analyst' is simply a 'BI Analyst', making analysis much easier.

In [38]:
#first check the variety of titles
df1['title'].value_counts()

title
Business Intelligence Analyst                             34
Power BI Developer                                        26
Data Analyst                                              21
Senior Power BI Developer                                 14
Tableau Developer                                         12
                                                          ..
Power BI Engineer (DAX, Power Query, Power BI Service)     1
Data Analyst - IT                                          1
EPIC Inpatient BI Developer                                1
Senior Business Intelligence Analyst                       1
Vice President, Analytics                                  1
Name: count, Length: 356, dtype: int64

In [47]:
title_keywords = {
    "data engineer": ["engineer", "engineering"],
    "data manager": ["manager", "management", "data director","president"],
    "data analyst": ["data analyst", "analytics", "analysis"],
    "bi analyst": ["bi analyst", "business intelligence analyst","business analyst"],
    "bi developer": ["bi developer", "business intelligence developer","developer","power bi"],
    "data scientist": ["scientist", "science"],
    "data architect": ["architect", "architecture"],
    "data specialist": ["specialist"],
    "data steward": ["steward", "librarian"],
    "data coordinator": ["coordinator", "data coordinator"],
    "data strategist": ["strategist", "strategy"],
    "data quality specialist": ["quality specialist", "quality assurance", "data quality"]
    }


In [48]:
def get_core_title(title):
    title_lower = title.lower()

    # Check each key in the dictionary
    for core_title, keywords in title_keywords.items():
        # If any keyword matches, return the core job title
        if any(keyword in title_lower for keyword in keywords):
            return core_title
    
    return "unknown"  #if no match found

In [49]:
df1["core_title"] = df1["title"].apply(get_core_title)
df1[["title", "core_title"]]

,title,core_title
0,Tableau Developer,bi developer
1,Power BI Co-Op,bi developer
2,Power BI Designer,bi developer
3,Power BI Developer,bi developer
4,Lead Power BI Developer,bi developer
...,...,...
795,Data Analyst - IT,data analyst
796,Data Visualization Specialist,data specialist
797,EPIC Inpatient BI Developer,bi developer
798,Senior Business Intelligence Analyst,bi analyst


In [50]:
df1.core_title.value_counts()

core_title
bi developer        291
data analyst        124
unknown             119
bi analyst          103
data engineer        54
data manager         40
data specialist      36
data strategist      14
data architect        7
data coordinator      3
Name: count, dtype: int64

In [51]:
df1['source']='glassdoor'

In [52]:
df1.info()

<class 'pandas.core.frame.DataFrame'>
Index: 791 entries, 0 to 799
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   title        791 non-null    object
 1   description  791 non-null    object
 2   link         791 non-null    object
 3   core_title   791 non-null    object
 4   source       791 non-null    object
dtypes: object(5)
memory usage: 37.1+ KB


## Text normalization